# CountZES — Spike Test & Playground (Google Colab + Drive Mount)

Notebook dùng để **thử nghiệm tự do (spike test)** trên ảnh bất kỳ (URL, ảnh có sẵn trong `MyDrive/CountZES/data/`, hoặc ảnh upload từ máy) với prompt class tự do.

Toàn bộ data, weights và output dùng chung thư mục Google Drive: `/content/drive/MyDrive/CountZES/`.

In [4]:
# [CELL-1] Thiết lập môi trường, clone/pull repo & cài thư viện
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

REPO_URL = "https://github.com/duylw/cv-advanced.git"
REPO_DIR = "/content/cv-advanced"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull origin main

!pip install -q opencv-python tqdm matplotlib timm einops supervision sentencepiece accelerate scipy pandas scikit-image addict yapf pycocotools gdown
!pip install -q "transformers>=4.31.0"
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!nvidia-smi

Cloning into '/content/cv-advanced'...
remote: Enumerating objects: 254, done.
remote: Counting objects: 100% (254/254), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 254 (delta 66), reused 253 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (254/254), 27.46 MiB | 20.07 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/cv-advanced
From https://github.com/duylw/cv-advanced
 * branch            main       -> FETCH_HEAD
Already up to date.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Sat Nov 21 18:08:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | V

In [5]:
# [CELL-2] Mount Google Drive (/content/drive/MyDrive/CountZES) chứa cả data & weights
import shutil
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CountZES"
DRIVE_DATA_DIR = os.path.join(DRIVE_PROJECT_DIR, "data")
DRIVE_WEIGHTS_DIR = os.path.join(DRIVE_PROJECT_DIR, "weights")
DRIVE_OUTPUTS_DIR = os.path.join(DRIVE_PROJECT_DIR, "outputs")

for d in [DRIVE_PROJECT_DIR, DRIVE_DATA_DIR, DRIVE_WEIGHTS_DIR, DRIVE_OUTPUTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Copy metadata (.json/.txt) từ repo sang Drive nếu Drive chưa có (idempotent, không ghi đè)
for root, _, files_in_dir in os.walk("/content/cv-advanced/data", followlinks=False):
    rel_root = os.path.relpath(root, "/content/cv-advanced/data")
    target_root = os.path.join(DRIVE_DATA_DIR, rel_root) if rel_root != "." else DRIVE_DATA_DIR
    os.makedirs(target_root, exist_ok=True)
    for fname in files_in_dir:
        src_f = os.path.join(root, fname)
        dst_f = os.path.join(target_root, fname)
        if not os.path.islink(src_f) and not os.path.exists(dst_f):
            shutil.copy2(src_f, dst_f)

os.makedirs("DSALVANet/checkpoints", exist_ok=True)
os.makedirs("GroundingDINO/weights", exist_ok=True)
os.makedirs("CounTR/output_allnew_dir", exist_ok=True)

def ensure_weight(weight_name, local_path, download_cmd=None):
    drive_path = os.path.join(DRIVE_WEIGHTS_DIR, weight_name)
    if not (os.path.exists(drive_path) and os.path.getsize(drive_path) > 0):
        if download_cmd:
            print(f"[Download -> Drive] {drive_path}")
            get_ipython().system(download_cmd.format(dst=drive_path))
        else:
            return
    else:
        print(f"[Drive OK] {drive_path}")
    if os.path.islink(local_path) or os.path.exists(local_path):
        os.remove(local_path)
    os.symlink(drive_path, local_path)

ensure_weight(
    "sam_vit_h_4b8939.pth",
    "data/sam_vit_h_4b8939.pth",
    "wget -q --show-progress -O '{dst}' https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
)
ensure_weight(
    "checkpoint_200.pth",
    "DSALVANet/checkpoints/checkpoint_200.pth",
    "gdown 1julzH9MJSK1xTGchb1r0CXdZ2wzF5-Kp -O '{dst}'"
)
ensure_weight(
    "groundingdino_swint_ogc.pth",
    "GroundingDINO/weights/groundingdino_swint_ogc.pth",
    "gdown 13rV6dzRwWCVZYWpnmiaVwRDIDC28d82g -O '{dst}'"
)
ensure_weight("FSC147.pth", "CounTR/output_allnew_dir/FSC147.pth")
ensure_weight("CARPK.pth", "CounTR/output_allnew_dir/CARPK.pth")

print("Project Drive Dir:", DRIVE_PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[Drive OK] /content/drive/MyDrive/CountZES/weights/sam_vit_h_4b8939.pth
[Download -> Drive] /content/drive/MyDrive/CountZES/weights/checkpoint_200.pth
Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1julzH9MJSK1xTGchb1r0CXdZ2wzF5-Kp

but Gdown can't. Please check connections and permissions.
[Download -> Drive] /content/drive/MyDrive/CountZES/weights/groundingdino_swint_ogc.pth
Downloading...
From (original): https://drive.google.com/uc?id=13rV6dzRwWCVZYWpnmiaVwRDIDC28d82g
From (redirected): https://drive.google.com/uc?id=13rV6dzRwWCVZYWpnmiaVwRDIDC28d82g&confirm=t&uuid=6e9d2bc5

In [6]:
# [CELL-3] Load toàn bộ 5 model 1 lần từ weights trên Drive
from types import SimpleNamespace
from countzes_core import load_pipeline_models

FSOC_BACKEND = "DSALVANet"  # "DSALVANet" hoặc "countr"
args = SimpleNamespace(
    data=DRIVE_DATA_DIR,
    outdir="spike_test",
    ckpt=os.path.join(DRIVE_WEIGHTS_DIR, "sam_vit_h_4b8939.pth"),
    sam_type="vit_h",
    ref_idx="00",
    visualize=True,
    fsoc=FSOC_BACKEND,
    weight=os.path.join(DRIVE_WEIGHTS_DIR, "checkpoint_200.pth"),
    countr_ckpt=os.path.join(DRIVE_WEIGHTS_DIR, "FSC147.pth"),
    fix_sses_inversion=False,
    single_instance_mode="paper",
)
models = load_pipeline_models(args)
print("Models loaded on:", models["device"])

ModuleNotFoundError: No module named 'groundingdino'

In [ ]:
# [CELL-4] Chọn ảnh thử nghiệm: từ URL, từ Google Drive (MyDrive/CountZES/data/), hoặc upload từ máy
import urllib.request

IMAGE_SOURCE = "url"   # "url" | "drive" | "upload"
CLASS_PROMPT = "person"  # Đổi tên object bất kỳ (vd: "cars", "apples", "cells", "bus")

# Dùng khi IMAGE_SOURCE == "url"
TEST_IMAGE_URL = "https://ultralytics.com/images/bus.jpg"

# Dùng khi IMAGE_SOURCE == "drive" (đường dẫn tương đối bên trong MyDrive/CountZES/data/)
DRIVE_REL_IMAGE = "vgg/Images/cells/00.png"

os.makedirs(os.path.join(DRIVE_OUTPUTS_DIR, "spike_inputs"), exist_ok=True)

if IMAGE_SOURCE == "url":
    TEST_IMAGE_PATH = os.path.join(DRIVE_OUTPUTS_DIR, "spike_inputs/test_sample.jpg")
    req = urllib.request.Request(TEST_IMAGE_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=30) as resp, open(TEST_IMAGE_PATH, "wb") as f:
        f.write(resp.read())
elif IMAGE_SOURCE == "drive":
    TEST_IMAGE_PATH = os.path.join(DRIVE_DATA_DIR, DRIVE_REL_IMAGE)
elif IMAGE_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    TEST_IMAGE_PATH = list(uploaded.keys())[0]
else:
    raise ValueError(f"Unsupported IMAGE_SOURCE: {IMAGE_SOURCE}")

assert os.path.exists(TEST_IMAGE_PATH), f"Không tìm thấy ảnh: {TEST_IMAGE_PATH}"
print(f"Test image -> {TEST_IMAGE_PATH} | Prompt: '{CLASS_PROMPT}'")

In [ ]:
# [CELL-5] Chạy nhanh CountZES và xem kết quả cuối
import cv2
import matplotlib.pyplot as plt
from countzes_core import run_countzes_on_image

vis_output, pred_count = run_countzes_on_image(
    image_path=TEST_IMAGE_PATH,
    filename=os.path.basename(TEST_IMAGE_PATH),
    class_name=CLASS_PROMPT,
    models=models,
    args=args,
)

print(f"==> Predicted Count for '{CLASS_PROMPT}': {pred_count}")
plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(vis_output, cv2.COLOR_BGR2RGB))
plt.title(f"CountZES ({args.fsoc}) | Prompt='{CLASS_PROMPT}' | Pred={pred_count}")
plt.axis("off")
plt.show()

In [ ]:
# [CELL-6] Soi chi tiết từng giai đoạn (CLIP map + GroundingDINO + Default vs A/B P2 flags)
from countzes_core import compute_clip_similarity_map, normalize_sim_map

src_img = cv2.imread(TEST_IMAGE_PATH)
img_rgb = cv2.cvtColor(src_img, cv2.COLOR_BGR2RGB)

sim_clip = normalize_sim_map(
    compute_clip_similarity_map(
        img_rgb, CLASS_PROMPT, models["clip_model"], models["clip_processor"], models["device"]
    )
)

det = models["grounding_dino"].predict_with_classes(
    image=src_img, classes=[CLASS_PROMPT], box_threshold=0.20, text_threshold=0.15
)
vis_det = img_rgb.copy()
if det is not None and len(det.xyxy) > 0:
    for (x0, y0, x1, y1), conf in zip(det.xyxy, det.confidence):
        cv2.rectangle(vis_det, (int(x0), int(y0)), (int(x1), int(y1)), (0, 255, 0), 2)
        cv2.putText(vis_det, f"{conf:.2f}", (int(x0), max(15, int(y0) - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

args_ab = SimpleNamespace(**vars(args))
args_ab.fix_sses_inversion = True
args_ab.single_instance_mode = "unit"
vis_ab, pred_ab = run_countzes_on_image(
    image_path=TEST_IMAGE_PATH,
    filename=os.path.basename(TEST_IMAGE_PATH),
    class_name=CLASS_PROMPT,
    models=models,
    args=args_ab,
)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
axes[0].imshow(sim_clip, cmap="jet"); axes[0].set_title("1. CLIP Similarity Map")
axes[1].imshow(vis_det); axes[1].set_title(f"2. GroundingDINO ({len(det.xyxy) if det is not None else 0} boxes)")
axes[2].imshow(cv2.cvtColor(vis_output, cv2.COLOR_BGR2RGB)); axes[2].set_title(f"3. Default (Paper) -> {pred_count}")
axes[3].imshow(cv2.cvtColor(vis_ab, cv2.COLOR_BGR2RGB)); axes[3].set_title(f"4. A/B P2 Fix -> {pred_ab}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()